# GraphRAG: Xelsis or AI News PDFs

Choose one profile in Section 1. It selects the input file, ontology, all four
LLM prompts, example questions, and output filenames.

| Profile | Input | Ontology and prompts |
|---|---|---|
| Xelsis | data/Xelsis.pdf, pages 5, 7–8, 14–16, 18–21, 28–30 | Machine operation, maintenance, troubleshooting |
| AI news | data/ai_news.pdf | AI copyright and governance |

The pipeline loads source text, extracts entities and relationships, builds
communities, summarizes them, and answers questions from those summaries.

> **Refactored implementation:** The explanatory flow below is preserved from
> the original notebook. The implementation now lives in four reusable modules:
> `graph_rag_schema.py`, `graph_rag_engine.py`, `graph_rag_services.py`, and
> `graph_rag_manager.py`. Notebook cells call those classes instead of repeating
> their implementation.

---
## 0. Install Dependencies

In [1]:
# %pip install -r requirements.txt

---
## 1. Select Profile & Import

Set PROFILE in the next cell to "Xelsis" or "AI news" **before importing src**.
It overrides the active_ontology default in ontology.yaml for this Python process
and selects the same profile in prompt.yaml.

**After changing profiles or editing either YAML file, restart the kernel and
run the notebook from the top.** Already-imported Pydantic models cannot switch
their allowed types safely in place. No YAML files are rewritten by the notebook.

In [2]:
import os
import sys
from pathlib import Path

PROFILE = "Xelsis"  # Choose "Xelsis" or "AI news"

loaded_schema = sys.modules.get("src.graph_rag_schema")
if loaded_schema is not None and getattr(loaded_schema, "ACTIVE_ONTOLOGY", None) != PROFILE:
    raise RuntimeError("Profile changed after import. Restart the kernel and run from the top.")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ontology.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT / "causalRAG"
if not (PROJECT_ROOT / "ontology.yaml").is_file():
    raise FileNotFoundError("Start the notebook from causalRAG or its parent directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from profile_config import load_profiles

PROFILES = load_profiles()

if PROFILE not in PROFILES:
    raise ValueError(f"Choose one of {list(PROFILES)}; got {PROFILE!r}")

os.environ["GRAPH_RAG_PROFILE"] = PROFILE
# Load all settings from the selected profile together.
PROFILE_CONFIG = PROFILES[PROFILE]
INPUT_TYPE = PROFILE_CONFIG["input_type"]
DATASET_FILE = PROJECT_ROOT / PROFILE_CONFIG["input_file"]
PDF_PAGES = PROFILE_CONFIG["pages"]
PDF_CHUNK_SIZE = PROFILE_CONFIG["pdf_chunk_size"]
PDF_CHUNK_OVERLAP = PROFILE_CONFIG["chunk_overlap"]
FORCE_REBUILD = PROFILE_CONFIG["force_rebuild"]

print(f"Profile: {PROFILE} | Input: {INPUT_TYPE.upper()} | File: {DATASET_FILE}")

Profile: Xelsis | Input: PDF | File: /home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/data/Xelsis.pdf


In [3]:
import nest_asyncio

from src import (
    ExtractedEntity,
    ExtractedRelationship,
    ExtractionResult,
    # GraphRAGExtractor,
    GraphRAGManager,
    GraphRAGQueryEngine,
    GraphRAGSchema,
    # GraphRAGService,
    GraphRAGStore,
)

if GraphRAGSchema.ACTIVE_ONTOLOGY != PROFILE:
    raise RuntimeError("Profile mismatch. Restart the kernel and run from the top.")

nest_asyncio.apply()
print(f"✅ Imports ready; ontology and prompts: {GraphRAGSchema.ACTIVE_ONTOLOGY}")

✅ Imports ready; ontology and prompts: Xelsis


---
## 2. Configuration

Set `USE_QWEN` in the next cell to switch the complete pipeline between the local Qwen model and OpenAI. Each profile has one output set in `output/`, shared across providers. Rebuilding overwrites that profile’s outputs.

| Provider | Extraction and community summaries | Final query synthesis |
|---|---|---|
| Qwen | `Qwen3.8-27B` through the local LiteLLM endpoint | `Qwen3.8-27B` |
| OpenAI | `gpt-5-nano` | `gpt-5.6-luna` |


In [4]:
LLM_CONFIG = PROFILE_CONFIG["llm"]
LLM_PROVIDER = LLM_CONFIG["provider"]
USE_QWEN = LLM_PROVIDER == "qwen"
EXTRACTION_MODEL = LLM_CONFIG["extraction_model"]
QUERY_MODEL = LLM_CONFIG["query_model"]
QWEN_BASE_URL = LLM_CONFIG["qwen_base_url"]

OUTPUT_PREFIX = PROFILE_CONFIG["output_prefix"]
OUTPUT_DIR = PROJECT_ROOT / "output"
CHECKPOINT_FILE = OUTPUT_DIR / f"{OUTPUT_PREFIX}_graph_store.pkl"
GRAPH_DATA_FILE = OUTPUT_DIR / f"{OUTPUT_PREFIX}_graph_data.json"
GRAPH_TEMPLATE_FILE = PROJECT_ROOT / "graph_template.html"
GRAPH_OUTPUT_FILE = OUTPUT_DIR / f"{OUTPUT_PREFIX}_graph.html"

MAX_PATHS_PER_CHUNK = LLM_CONFIG["max_paths_per_chunk"]
NUM_WORKERS = LLM_CONFIG["num_workers"]
MAX_CLUSTER_SIZE = LLM_CONFIG["max_cluster_size"]
REQUEST_TIMEOUT = LLM_CONFIG["request_timeout"]
REQUEST_MAX_RETRIES = LLM_CONFIG["request_max_retries"]

manager = GraphRAGManager(
    provider=LLM_PROVIDER,
    extraction_model=EXTRACTION_MODEL,
    query_model=QUERY_MODEL,
    qwen_base_url=QWEN_BASE_URL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
    max_cluster_size=MAX_CLUSTER_SIZE,
    request_timeout=REQUEST_TIMEOUT,
    request_max_retries=REQUEST_MAX_RETRIES,
)

EXTRACTION_LLM = manager.extraction_llm
QUERY_LLM = manager.query_llm

print(
    f"✅ Using {LLM_PROVIDER}: {EXTRACTION_LLM.model} for extraction, "
    f"{QUERY_LLM.model} for querying"
)


✅ Using qwen: Qwen3.8-27B for extraction, Qwen3.8-27B for querying


---
## 3. Ontology

The selected profile in ontology.yaml defines allowed entity and relationship
types. The same definitions drive the extraction prompt and Pydantic validation.
Xelsis and AI news have separate vocabularies.

In [5]:
ENTITY_TYPES = GraphRAGSchema.ENTITY_TYPES
RELATION_TYPES = GraphRAGSchema.RELATION_TYPES

print("✅ Ontology:")
print(f"   Entity types:       {ENTITY_TYPES}")
print(f"   Relationship types: {RELATION_TYPES}")

✅ Ontology:
   Entity types:       ('DEVICE', 'COMPONENT', 'CONTROL', 'FEATURE', 'SETTING', 'RESOURCE', 'DRINK', 'CONDITION', 'ACTION', 'DIAGNOSTIC_CASE')
   Relationship types: ('HAS_COMPONENT', 'HAS_FEATURE', 'HAS_SETTING', 'ACCESSED_VIA', 'ACTS_ON', 'USES', 'PRODUCES', 'HAS_STEP', 'PRECEDES', 'REQUIRES_CONDITION', 'MAY_CAUSE', 'PREVENTS', 'HAS_SYMPTOM', 'HAS_POSSIBLE_CAUSE', 'HAS_REMEDY')


---
## 4. Extraction Prompt

The selected profile in prompt.yaml supplies the extraction instructions.
Allowed types and their descriptions are inserted from ontology.yaml.
The LLM returns descriptions alongside entities and relationships so community
summaries can retain context, conditions, and qualifications.

In [6]:
KG_TRIPLET_EXTRACT_TMPL = GraphRAGSchema.extraction_prompt()

print("✅ Extraction prompt ready")
print(f"\nPreview (first 300 chars):\n{KG_TRIPLET_EXTRACT_TMPL[:300]}...")

✅ Extraction prompt ready

Preview (first 300 chars):
-Goal-
Extract operation, maintenance, and troubleshooting knowledge from the
supplied Saeco Xelsis manual excerpt.
Extract up to {max_knowledge_triplets} entity-relation triplets.

-Allowed Entity Types-
- DEVICE: The appliance being described, such as the Saeco Xelsis coffee machine.
- COMPONENT: ...


---
## 5. Pydantic Extraction Models

Structured output is validated using the selected ontology:

- ExtractedEntity: name, type, description
- ExtractedRelationship: source, target, relation, description
- ExtractionResult: the entity and relationship lists

EntityType and RelationType are runtime Literal types loaded from YAML.
Pydantic rejects labels outside the selected profile.

In [7]:
print("✅ Pydantic extraction models:")
print("  ", ExtractedEntity.__name__)
print("  ", ExtractedRelationship.__name__)
print("  ", ExtractionResult.__name__)

✅ Pydantic extraction models:
   ExtractedEntity
   ExtractedRelationship
   ExtractionResult


---
## 6. GraphRAGExtractor

This is the core extraction component. It sends each text chunk to the LLM
with our ontology-constrained prompt, parses the response, and stores the
extracted entities and relationships as structured objects on each node.

### Why build a custom extractor?

LlamaIndex's built-in `SchemaLLMPathExtractor` extracts entity/relationship labels
but **drops descriptions**. By building our own extractor:

- Every `EntityNode` carries a `entity_description` property
- Every `Relation` carries a `relationship_description` property
- These flow through to community summaries, making them far richer

The extractor runs **asynchronously** with `num_workers=2` — processing 2 chunks
in parallel while limiting pressure on long structured-output API requests.


In [8]:
kg_extractor = manager.extractor

print(f"✅ {type(kg_extractor).__name__} ready")
print(f"   Parallel workers: {kg_extractor.num_workers}")
print(f"   Maximum paths per chunk: {kg_extractor.max_paths_per_chunk}")

✅ GraphRAGExtractor ready
   Parallel workers: 2
   Maximum paths per chunk: 20


---
## 7. GraphRAGStore

`GraphRAGStore` extends LlamaIndex's `SimplePropertyGraphStore` with two additional
capabilities: **community detection** and **community summary generation**.

By bundling these into the store itself, the pipeline stays clean — after building
the index you just call `graph_store.build_communities()` and everything is handled.

### How community detection works here

1. Convert the property graph to a NetworkX graph
2. Run hierarchical Leiden to find entity clusters
3. For each cluster, collect all entities (+ descriptions) and relationships (+ descriptions)
4. Ask the LLM to write a briefing note for each cluster

The descriptions captured during extraction make these briefings significantly
richer than if we'd only stored bare labels.


In [9]:
# The manager creates this store when a graph is built or loaded.
print(f"✅ Graph store class ready: {GraphRAGStore.__name__}")

✅ Graph store class ready: GraphRAGStore


---
## 8. GraphRAGQueryEngine

The query engine uses a two-step approach:

1. **Per-community answering** — ask the LLM to answer the question from each
   community summary independently. If a summary isn't relevant, the LLM says so
   and we skip it. This avoids polluting the final answer with irrelevant content.

2. **Aggregation** — combine all relevant partial answers into one final,
   non-redundant response using `QUERY_LLM` (the stronger model).


In [10]:
# The query engine is instantiated after a graph has been built or loaded.
print(f"✅ Query engine class ready: {GraphRAGQueryEngine.__name__}")

✅ Query engine class ready: GraphRAGQueryEngine


## 9. Inspect the Selected PDF

Both profiles use text PDFs. Supply `data/ai_news.pdf` for AI news. Pages without extractable text are skipped and reported.

In [11]:
if not DATASET_FILE.is_file():
    raise FileNotFoundError(f"PDF not found: {DATASET_FILE}. Supply a text-based PDF for {PROFILE}.")
print(f"PDF: {DATASET_FILE.name}")
print(f"Pages: {PDF_PAGES if PDF_PAGES is not None else 'all'}")
print(f"Chunk size: {PDF_CHUNK_SIZE} tokens | Overlap: {PDF_CHUNK_OVERLAP}")

PDF: Xelsis.pdf
Pages: 5, 7-8, 14-16, 18-21, 28-30
Chunk size: 1000 tokens | Overlap: 100


## 10. Load PDF Chunks

Both profiles use manager.load_pdf_documents(). Each chunk retains its file,
page, extracted-text line range, and source excerpt. Every extracted triple
inherits this provenance. Line ranges refer to the source chunk, not an exact
sentence supporting the triple. Chunks never cross page boundaries.

In [12]:
if GraphRAGSchema.ACTIVE_ONTOLOGY != PROFILE:
    raise RuntimeError("Profile mismatch. Restart the kernel and run from the top.")

nodes = manager.load_pdf_documents(
    DATASET_FILE,
    pages=PDF_PAGES,
    chunk_size=PDF_CHUNK_SIZE,
    chunk_overlap=PDF_CHUNK_OVERLAP,
)

if not nodes:
    raise ValueError("The selected input produced no documents.")
print(f"✅ Loaded {len(nodes)} input nodes for {PROFILE}")

Loaded 13 page(s) from '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/data/Xelsis.pdf' and created 13 chunks
✅ Loaded 13 input nodes for Xelsis


In [13]:
SAMPLE_INDEX = 0  # Choose any loaded document/chunk index
if not 0 <= SAMPLE_INDEX < len(nodes):
    raise IndexError(f"SAMPLE_INDEX must be between 0 and {len(nodes) - 1}")
print(nodes[SAMPLE_INDEX].metadata)
print(nodes[SAMPLE_INDEX].get_content())

{'title': '4219.450.3076.2  Xelsis 2.0 Full DFU EU9', 'source': '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/data/Xelsis.pdf', 'file_name': 'Xelsis.pdf', 'page_number': 5, 'chunk_index': 1, 'provenance': [{'source': '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/data/Xelsis.pdf', 'file_name': 'Xelsis.pdf', 'page_number': 5, 'page_chunk_index': 1, 'scope': 'source_chunk', 'excerpt': "Machine overview (Fig. A)\nA1 Lid of bean hopper\nA2 Lid of pre-ground coffee\ncompartment\nA3 Cup tray\nA4 Control panel\nA5 Socket for cord\nA6 Main switch\nA7 Drip tray\nA8 Drip tray release grip\nA9 HygieSteam container\nA10 HygieSteam cover with milk tube\nholder\nA11 Height-adjustable coffee and milk\ndispensing spout\nA12 Internal cappuccinatore\nA13 Power cord with plug\nA14 Grind setting knob (to learn more,\nvisit www.saeco.com/care )\nA15 Bean hopper\nA16 Service door\nA17 Coffee funnel\nA18 Coffee residues drawer\nA19 Brew group (to learn more, visit\nwww.saeco.com/ca

---
## 11. Build the Knowledge Graph 

Now we wire everything together and run the extraction pipeline.

`PropertyGraphIndex` handles the full workflow:
1. Passes each chunk to `GraphRAGExtractor`
2. The extractor calls the LLM with our ontology-constrained prompt
3. Parsed entities and relationships are stored in `GraphRAGStore`

-> This is the most time-consuming step!


Checkpoint, JSON, and HTML live in `output/`, with one file of each kind per
profile (`xelsis_*` or `ai_news_*`), shared across providers.

Set FORCE_REBUILD = True after changing the source, PDF pages, chunk settings,
provider, ontology, or prompts. Restart the kernel after YAML changes.
Migrated legacy checkpoints retain only their recorded source information.
Rebuild from PDFs for line ranges and all source occurrences.

In [15]:
CHECKPOINT_FILE.exists()

False

In [16]:
REBUILD_GRAPH = FORCE_REBUILD or not CHECKPOINT_FILE.exists()

if REBUILD_GRAPH:
    # Community detection is kept for Section 12 so the original flow remains clear.
    graph_store = manager.build_knowledge_graph(
        documents=nodes,
        build_communities=False,
    )
else:
    graph_store = manager.load_knowledge_graph(CHECKPOINT_FILE)

Building knowledge graph; LLM extraction may take several minutes...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
# Optional extra LLM call on one loaded document/chunk.
RUN_EXTRACTION_TEST = False
if RUN_EXTRACTION_TEST:
    extraction_test = await manager.atest_extraction(document_index=SAMPLE_INDEX)

In [18]:
entities_by_type = manager.print_unique_entities()


ACTION (73)
  Activate AquaClean Filter
  Activate AquaClean water filter
  Adjust Grind Setting
  Adjust Grinder to Finer Setting
  Adjust drink settings
  Brew 5 Cups for Self-Adjustment
  Brew Group Clean
  Brewing procedure
  Clean HygieSteam container
  Clean HygieSteam parts
  Clean Internal Cappuccinatore
  Clean Milk System Daily
  Clean and Lubricate Brew Group
  Clean brew group under tap
  Clean brew group with tablets
  Clean coffee dispensing spout
  Clean drip tray
  Clean internal cappuccinatore
  Clean machine front
  Clean milk container
  Clean milk container parts
  Close Service Door
  Connect to Wi-Fi
  Descale machine
  Descale the machine
  Descaling procedure
  Determine Water Hardness
  Disassemble milk container
  Dislodge clogged coffee
  Empty Drip Tray
  Empty coffee grounds container
  Fill water tank and bean hopper
  Follow On-Screen Steps
  Immerse Test Strip
  Lubricate brew group
  Open service door
  Perform Deep Milk Clean
  Position Hook
  Positio

In [19]:
# Choose an entity that actually exists in the selected graph.
ENTITY_TO_INSPECT = next(
    (name for names in entities_by_type.values() for name in names),
    None,
)
entity_details = (
    manager.inspect_entity(ENTITY_TO_INSPECT)
    if ENTITY_TO_INSPECT is not None else None
)

Node: 'Activate AquaClean Filter'  label='ACTION'
Source: /home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/data/Xelsis.pdf
Title: 4219.450.3076.2  Xelsis 2.0 Full DFU EU9

=== Article text ===
Machine overview (Fig. A)
A1 Lid of bean hopper
A2 Lid of pre-ground coffee
compartment
A3 Cup tray
A4 Control panel
A5 Socket for cord
A6 Main switch
A7 Drip tray
A8 Drip tray release grip
A9 HygieSteam container
A10 HygieSteam cover with milk tube
holder
A11 Height-adjustable coffee and milk
dispensing spout
A12 Internal cappuccinatore
A13 Power cord with plug
A14 Grind setting knob (to learn more,
visit www.saeco.com/care )
A15 Bean hopper
A16 Service door
A17 Coffee funnel
A18 Coffee residues drawer
A19 Brew group (to learn more, visit
www.saeco.com/care )
A20 Coffee grounds container
A21 Water tank
A22 Lid of water tank
A23 'Drip tray full' indicator
A24 Drip tray cover
Accessories
A25 Milk container
A26 Milk tube
A27 Cleaning brush
A28 Grease tube
A29 AquaClean filter
A30 Measurin

---
## 12. Build Communities & Generate Summaries

Community summaries use the selected profile's community_summary template in
prompt.yaml. Xelsis summaries emphasize maintenance and troubleshooting;
AI news summaries emphasize copyright and governance.

In [20]:
if REBUILD_GRAPH:
    summaries = graph_store.build_communities(
        summary_llm=EXTRACTION_LLM,
        max_cluster_size=MAX_CLUSTER_SIZE,
    )
    manager.save_knowledge_graph(CHECKPOINT_FILE)
else:
    summaries = graph_store.get_community_summaries()

print(f"\n✅ {len(summaries)} community summaries ready for querying")

Running community detection...
Graph has 269 nodes, 263 edges
Found 62 communities


/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/.venv/lib/python3.12/site-packages/graspologic/partition/leiden.py:607: UserWarning: Leiden partitions do not contain all nodes from the input graph because input graph contained isolate nodes.
  warnings.warn(


  Community 0: The Saeco Xelsis 2.0 (Model 4219.450.3076.2) requires users to read the safety booklet before first ...
  Community 1: The Saeco Xelsis requires weekly maintenance of its milk system, specifically the HygieSteam contain...
  Community 2: The HygieSteam feature is a named capability that acts on the Milk System to clean it, specifically ...
  Community 3: The Saeco Xelsis utilizes a 100% ceramic grinder with adjustable coarse-to-fine settings and a heigh...
  Community 4: The Saeco Xelsis brew group is a physical component that requires regular maintenance to prevent dir...
  Community 5: To configure the Saeco Xelsis, users must first tap the Settings icon and swipe left to right to loc...
  Community 6: To operate the Saeco Xelsis, users must first fill the water tank and bean hopper with coffee beans,...
  Community 7: To determine regional water hardness for the Saeco Xelsis, use the supplied Water Hardness Test Stri...
  Community 8: The Saeco Xelsis requires the Aqu

## 13. Explore in Streamlit

Run `.venv/bin/streamlit run app.py -- xelsis` or `.venv/bin/streamlit run app.py -- ai_news`
from causalRAG. The app loads the profile checkpoint, shows the interactive graph,
and includes Query the System. Profile and LLM settings live in profiles.yaml.
The following cell remains an optional standalone HTML export.


In [21]:
# A running kernel may still hold classes imported before code changes.
if not hasattr(graph_store, "community_members"):
    raise RuntimeError("Restart the kernel and run from the top to export community assignments.")

manager.visualize(
    graph_data_file=GRAPH_DATA_FILE,
    template_file=GRAPH_TEMPLATE_FILE,
    output_file=GRAPH_OUTPUT_FILE,
)

Graph data exported to '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/output/xelsis_graph_data.json'
Nodes: 269 | Edges: 266 | Communities: 62
Visualization saved to '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/output/xelsis_graph.html'


PosixPath('/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/output/xelsis_graph.html')

---
## 14. Query the System

Example questions follow the selected input profile. Answers use community
summaries and the matching community_answer and aggregation prompts from
prompt.yaml.

In [22]:
query_engine = GraphRAGQueryEngine(
    graph_store=graph_store,
    community_llm=EXTRACTION_LLM,
    llm=QUERY_LLM,
)

print("✅ Query engine ready")

✅ Query engine ready


In [23]:
q1 = PROFILE_CONFIG["questions"][0]
print(f"Query: {q1}")
print("=" * 70)
print(query_engine.custom_query(q1))

Query: What are the possible causes of watery coffee and the remedy for each cause?
Based on the provided community answers, here are the possible causes of watery coffee on the Saeco Xelsis and their corresponding remedies:

1.  **Dirty or unlubricated brew group**
    *   **Remedy:** Remove the brew group, rinse it under tap water, dry it, and lubricate the moving parts.

2.  **Machine currently performing its self-adjustment procedure**
    *   **Remedy:** Brew five cups of coffee to allow the self-adjustment procedure to complete.

3.  **Grinder set too coarse**
    *   **Remedy:** Adjust the grinder to a finer (lower) setting. After making this adjustment, brew 2–3 drinks to taste the difference and verify the improvement.

*Note: The supplied answers do not provide specific maintenance intervals, additional warnings, or a complete step-by-step procedure for disassembling or lubricating the brew group beyond the general actions listed above.*


In [24]:
q2 = PROFILE_CONFIG["questions"][1]
print(f"Query: {q2}")
print("=" * 70)
print(query_engine.custom_query(q2))

Query: What maintenance does the brew group need, and how often?
Based on the provided community answers, the maintenance requirements for the Saeco Xelsis brew group are as follows:

**1. Cleaning**
*   **Frequency:** One source specifies **weekly**. Other sources state that the frequency is not specified in their available information, or that cleaning is required "if it is dirty."
*   **Procedure:**
    1.  Open the service door.
    2.  Remove the coffee residues drawer.
    3.  Access the menu via **STATUS -> PERFORMANCE -> BREW GROUP CLEAN** (or select "Brew group clean" from the Status menu).
    4.  Follow on-screen instructions to remove the brew group.
    5.  Rinse the brew group under tap water to remove coffee residues and oil.
    6.  Dry the component.
*   **Note on Agents:** One source mentions using "specific Philips coffee oil remover tablets," while others specify rinsing under tap water. The use of tablets is not confirmed by the other sources.

**2. Lubrication**
*

In [25]:
q3 = PROFILE_CONFIG["questions"][2]
print(f"Query: {q3}")
print("=" * 70)
print(query_engine.custom_query(q3))

Query: When should I replace and activate the AquaClean filter?
**When to Replace**
Replace the AquaClean filter under the following conditions:
*   **Time-based:** At least every 3 months, even if the machine does not indicate it is due.
*   **Usage-based:** When the filter reaches the end of its life, defined as 95 liters of usage or when the AquaClean status indicator drops to 0%.
*   **Prompt-based:** When the Maintenance Dashboard triggers a prompt indicating the filter has expired or that three months have passed since the last change.

**Prerequisites**
*   **Limescale Condition:** The machine must be in a limescale-free condition before starting use with a new filter. This is a strict prerequisite.

**How to Activate**
*   **Timing:** Activate the filter immediately after installing the new one.
*   **Method:** The machine does not automatically detect the new filter. You must manually activate it via the **Status menu**.
*   **Duration:** The activation process takes approxima

In [ ]:
# Replace this with your own question about the selected dataset.
your_question = PROFILE_CONFIG["questions"][3]
print(f"Query: {your_question}")
print("=" * 70)
print(query_engine.custom_query(your_question))

In [ ]:
CHECKPOINT_FILE.exists()

---
## 15. Pipeline Reference

| Component | Purpose |
|---|---|
| PROFILE | Select input, vocabulary, prompts, questions, and output names |
| ontology.yaml | Allowed entity and relationship types |
| prompt.yaml | Extraction, summary, answering, and synthesis instructions |
| GraphRAGManager | Load PDFs, build, persist, inspect, and query |
| GraphRAGStore | Entity graph and community summaries |
| D3.js visualization | Interactive graph in the selected HTML output |